# ITASORL - end-to-end experiments on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iLevyTate/ItaSoRL/blob/main/notebooks/colab_gpu.ipynb)

## Run it (3 steps)

1. **Runtime -> Change runtime type -> GPU**, then **Save**.
2. **Runtime -> Run all** (top menu).
3. Wait for the run to finish, then read the **`SUMMARY.md`** printed near the
   bottom. A `bundle.zip` with everything downloads automatically.

That's it. The default profile is **`quick`** (~25 min): it runs the whole
pipeline at reduced scale and prints a green summary, so you can confirm the
project works end-to-end on your machine right now. To reproduce the real
research results, change `RUN_PROFILE` in the form below to `full` or a `bv3_*`
profile (see the table at the bottom).

## If Colab disconnects or the run stops early

Don't worry and don't start over. **Just run the notebook again** (Runtime ->
Run all). Every step is checkpointed as it finishes, so the run picks up exactly
where it left off - at most one step repeats. You'll see lines like
`--- resume skip expA_l1 (already ok) ---` confirming it skipped finished work.

Leave `USE_DRIVE` ticked (below) and progress is also mirrored to your Google
Drive, so a run survives even a full VM reset: reopen the notebook on a fresh
runtime, Run all, and it continues from Drive.

## What this measures (one line)

Does a from-scratch survival agent *incidentally* encode which world it lives in
(authentic vs a surrogate whose drag differs), readable by a linear probe but
never rewarded? The pre-registered bar is **probe AUROC >= 0.65**.

In [ ]:
# @title Configure, then run this cell { display-mode: "form" }
# Leave everything at its default for a quick end-to-end check.
RUN_PROFILE = "quick"  # @param ["quick", "full", "bv3_regime", "bv3_regime_n10", "bv2_ceiling", "bv3_ceiling", "bv3_ceiling_n10", "b2_only", "b2_seed0", "experiments_no_b2"]
USE_DRIVE = True  # @param {type:"boolean"}
FORCE_FRESH = False  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}
RESUME_RUN_DIR = ""  # @param {type:"string"}

from pathlib import Path  # noqa: E402

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    REPO_DIR = "/content/ITASORL"
except ImportError:
    IN_COLAB = False
    REPO_DIR = str(Path.cwd().resolve())
    if not (Path(REPO_DIR) / "scripts" / "run_e2e.py").is_file():
        REPO_DIR = str(Path(REPO_DIR).parent)

REPO_URL = "https://github.com/iLevyTate/ItaSoRL.git"
DRIVE_RESULTS = "/content/drive/MyDrive/ITASORL_results"
RESUME_RUN_DIR = RESUME_RUN_DIR.strip()
USE_DRIVE = USE_DRIVE and IN_COLAB

print(f"profile={RUN_PROFILE}  use_drive={USE_DRIVE}  force_fresh={FORCE_FRESH}  branch={BRANCH}")
if RUN_PROFILE != "quick":
    print(f"Note: {RUN_PROFILE!r} is a full-scale run (can take hours). If Colab "
          "disconnects, just Run all again - it resumes automatically.")
if BRANCH != "main":
    print(f"WARNING: running branch {BRANCH!r}, not main. For testing unmerged "
          "code only; do not treat the results as replication runs.")

## 1. Set up (Drive, code, dependencies, GPU)

Clones the repo, installs dependencies, and reports the GPU. `USE_DRIVE=True`
asks for Google Drive access once per fresh runtime (Google requires the popup on
each new VM). It buys crash-proof runs: progress mirrors to
`MyDrive/ITASORL_results` and a fresh VM resumes automatically.

Prefer no popup? Untick `USE_DRIVE` above: everything still runs and the bundle
still downloads at the end, but a full VM reset would lose progress (resume then
works only within the same runtime).

In [ ]:
import os, subprocess, sys

def sh(cmd):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RESULTS, exist_ok=True)

if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        sh(f"cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}")
    else:
        sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}")
else:
    print(f"Local mode: using existing checkout at {REPO_DIR}")
os.chdir(REPO_DIR)
sh("git log -1 --oneline")

sh(f"{sys.executable} -m pip install -q -r requirements-dev.txt")

import torch  # noqa: E402
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU detected. Runtime -> Change runtime type -> GPU, then re-run this cell.")
    print("CPU-only still works for the 'quick' profile; the bigger profiles are much slower.")

In [ ]:
# Keep-alive (Colab only): nudges the session so it stays awake during a long
# run. Leave the browser tab open. Harmless for the quick profile.
if IN_COLAB:
    from IPython.display import display, Javascript

    display(Javascript("""
    (function () {
      const intervalMs = 60000;
      setInterval(() => {
        fetch('/gen_' + Date.now()).catch(() => {});
      }, intervalMs);
      console.log('ITASORL keep-alive active every', intervalMs / 1000, 's');
    })();
    """))
    print("Keep-alive enabled. Leave this tab open until the run finishes.")
else:
    print("Local mode: keep-alive not needed.")

## 2. Run the experiments

One command. Profile flags, checkpointing, Drive mirroring, and auto-resume of an
unfinished run are all handled inside `scripts/run_e2e.py`; the printed `$ ...`
line shows the exact invocation.

If this cell stops (disconnect, timeout, or the runtime is recycled), **just run
it again** - it continues from the last finished step. `FORCE_FRESH` starts over
from scratch; `RESUME_RUN_DIR` continues one specific run directory (advanced).

In [ ]:
import os, subprocess, sys, time  # noqa: F811

cmd = [sys.executable, "scripts/run_e2e.py", "--profile", RUN_PROFILE]
if USE_DRIVE:
    cmd += ["--drive-sync", DRIVE_RESULTS]
if FORCE_FRESH:
    cmd += ["--force-fresh"]
if RESUME_RUN_DIR:
    cmd += ["--resume", RESUME_RUN_DIR]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
print("$", " ".join(cmd), flush=True)
t0 = time.time()
rc = subprocess.run(cmd, cwd=REPO_DIR, env=env).returncode
print(f"Wall time: {(time.time() - t0) / 60:.1f} min")
if rc != 0:
    raise RuntimeError(
        f"Run exited with code {rc}. Progress is checkpointed - re-run this cell "
        "(or Runtime -> Run all after a disconnect) to continue where it left off.")

## 3. Read the results

Prints `SUMMARY.md` (the plain-English verdict), compares B-v2 survival @ drift
0.45 to the canonical artifacts, and - when the profile dumped recurrent states -
recomputes the world-identity probe under level vs dispersion features.

If you run this before the run above has finished, it just tells you so; nothing
breaks.

In [ ]:
import subprocess, sys  # noqa: F811
from pathlib import Path

RUN_DIR = None
BUNDLE = None
_ptr = Path(REPO_DIR) / "results" / "LATEST_RUN.txt"
if not _ptr.is_file():
    print("No completed run yet. Run the '2. Run the experiments' cell above first.")
else:
    RUN_DIR = Path(_ptr.read_text().strip())
    _summary = RUN_DIR / "SUMMARY.md"
    BUNDLE = RUN_DIR / "bundle.zip"
    print("Run directory:", RUN_DIR)
    if not _summary.is_file():
        print("\nThis run hasn't finished (no SUMMARY.md yet). Re-run the run cell "
              "to continue it, then run this cell again.")
    else:
        print("\n" + "=" * 72)
        print(_summary.read_text())
        print("=" * 72)

        _cmp = Path(REPO_DIR) / "scripts" / "compare_expB2_artifacts.py"
        if _cmp.is_file():
            print("\nB-v2 artifact comparison (survival @ drift 0.45)")
            subprocess.run([sys.executable, str(_cmp), "--run", str(RUN_DIR)],
                           cwd=REPO_DIR, check=False)

        STATES = RUN_DIR / "artifacts" / "states"
        if STATES.is_dir():
            print("\nVariance & selectivity re-analysis (no GPU)")
            subprocess.run([sys.executable, "scripts/reanalyze_expB2_states.py", str(STATES)],
                           cwd=REPO_DIR, check=False)
        else:
            print("\n(no dumped states in this run; variance/selectivity re-analysis skipped)")

In [ ]:
# Download the bundle. If USE_DRIVE was on, it was already mirrored to Drive
# during the run, so no extra copying is needed here.
if not BUNDLE or not BUNDLE.is_file():
    print("No bundle to download yet - finish the run cell above, then re-run "
          "the results cell and this one.")
else:
    print("Bundle:", BUNDLE)
    if USE_DRIVE:
        print("Drive mirror:", f"{DRIVE_RESULTS}/{RUN_DIR.name}/bundle.zip")
    if IN_COLAB:
        from google.colab import files
        files.download(str(BUNDLE))
        print("Browser download started for bundle.zip")

## Reference

### Run profiles
`quick` is the default and proves the pipeline works. The rest reproduce the
research results and take much longer - all of them auto-resume if interrupted.

| Profile | ~Time | What it runs |
|---------|-------|--------------|
| `quick` | ~25 min | **default** - whole pipeline at reduced scale (smoke test) |
| `full` | ~4 hr | full replication: A + B + B-v2 (300 upd, 3 seeds) |
| `bv3_regime` | ~3.7 hr | B-v3 threshold attempt - per-episode drag *regime* |
| `bv3_regime_n10` | ~11.5 hr | B-v3 power extension - fresh seeds 0-9; resume across sessions |
| `bv2_ceiling` | ~3.7 hr | B-v2 (ar1) capacity ceiling (`--sysid-aux`) |
| `bv3_ceiling` | ~3.7 hr | B-v3 capacity ceiling (regime + `--sysid-aux`) |
| `bv3_ceiling_n10` | ~11.5 hr | B-v3 ceiling power extension - sysid-aux at seeds 0-9; tightens the capacity reference |
| `b2_only` | ~3.7 hr | B-v2 only, 3 seeds |
| `b2_seed0` | ~75 min | B-v2 only, seed 0 (replication-gap diagnostic) |
| `experiments_no_b2` | ~15 min | everything except B-v2 |

### What's in the bundle
| File | Purpose |
|------|---------|
| `SUMMARY.md` | Plain-English outcome. **Read this first.** |
| `status.json` | Live step + last line (updated during the run) |
| `manifest.json` | Step timings, status, artifact index |
| `combined.log` | Full stdout (updated live during the run) |
| `steps/*.json` | Parsed metrics (AUROC per drift, etc.) |
| `artifacts/` | Figures + `expB2_results.json` |
| `artifacts/cells/` | Per (drift, seed) checkpoints (resume granularity) |
| `artifacts/states/` | Dumped recurrent states for offline re-probing |

**Watch progress while a long run is going:**
- Local: `python scripts/watch_run.py --follow`
- Colab + Drive: open `MyDrive/ITASORL_results/<run>/combined.log`

Unzip the bundle locally and open `SUMMARY.md` to decide whether the organism
encoded world identity.